### Feature Extraction

####  *AUTHOR:* Ehsan Farahbakhsh
####  *CONTACT:* e.farahbakhsh@sydney.edu.au
####  *DATE last modified:* 05/07/2025

In [1]:
from ipywidgets import interact
import os

import cartopy.crs as ccrs
import cmcrameri.cm as ccm
import gplately
from gplately import PlateReconstruction, PlotTopologies
from gplately.tools import plate_isotherm_depth
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
import matplotlib.pyplot as plt

from lib.feature_extraction import *
from lib.slab_dip import calculate_slab_dip
from lib.water_thickness import calculate_water_thickness

from parameters import parameters

In [2]:
# Timespan for analysis
temporal_resolution = parameters["temporal_resolution"]
time_min = parameters["timespan"]["min"]
time_max = parameters["timespan"]["max"]
time_steps = range(time_min, time_max + temporal_resolution, temporal_resolution)

grid_resolution = parameters["grid_resolution"]

plate_model_dir = parameters["plate_model_dir"]
inputs_dir = parameters["inputs_dir"]
outputs_dir = parameters["outputs_dir"]
if not os.path.exists(outputs_dir):
    os.makedirs(outputs_dir, exist_ok=True)

subduction_data_filename = parameters["subduction_data_filename"]
subduction_data_filename = os.path.join(outputs_dir, subduction_data_filename)

agegrid_dir = os.path.join(inputs_dir, "SeafloorAge")
spreadrate_dir = os.path.join(inputs_dir, "SpreadingRate")
sedthick_dir = os.path.join(inputs_dir, "SedimentThickness")
carbonate_dir = os.path.join(inputs_dir, "CarbonateThickness")
co2_dir = os.path.join(inputs_dir, "CrustalCO2")

n_jobs = 20

In [3]:
# muller22
rotation_model = [
    plate_model_dir+"/Rotations/1000_0_rotfile_MantleOpt.rot",
    plate_model_dir+"/Rotations/1000_0_rotfile_Merdith_etal_opt.rot",
    plate_model_dir+"/Rotations/lat_lon_velocity_domain_30_60.gpml",
    plate_model_dir+"/Rotations/no_net_rotation_model.rot",
]

topology_features = [
    plate_model_dir+"/Topologies/250-0_plate_bounds.gpml",
    plate_model_dir+"/Topologies/410-250_plate_bounds.gpml",
    plate_model_dir+"/Topologies/1000-410-Convergence.gpml",
    plate_model_dir+"/Topologies/1000-410-Divergence.gpml",
    plate_model_dir+"/Topologies/1000-410-Topologies.gpml",
    plate_model_dir+"/Topologies/1000-410-Transforms.gpml",
    plate_model_dir+"/Topologies/TopologyBuildingBlocks.gpml",
]

static_polygons = plate_model_dir+"/shapes_static_polygons_Merdith_etal.gpml"
coastlines = plate_model_dir+"/shapes_coastlines_Merdith_etal.gpmlz"
continents = plate_model_dir+"/shapes_continents.gpml"
COBs = plate_model_dir+"/COB_polygons_and_coastlines_combined_1000_0_Merdith_etal.gpml"

plate_reconstruction = PlateReconstruction(
    rotation_model=rotation_model,
    topology_features=topology_features,
    static_polygons=static_polygons,
)

gplot = PlotTopologies(
    plate_reconstruction=plate_reconstruction,
    coastlines=coastlines,
    continents=continents,
    COBs=COBs,
)

### Feature Extraction

In [4]:
if os.path.isfile(subduction_data_filename):
    subduction_data = pd.read_csv(subduction_data_filename)
else:
    subduction_data = run_calculate_convergence(
        min_time=time_min,
        max_time=time_max,
        temporal_resolution=temporal_resolution,
        rotation_model=rotation_model,
        topology_features=topology_features,
        static_polygons=static_polygons,
        n_jobs=n_jobs,
        verbose=True,
    )

    subduction_data = run_coregister_ocean_rasters(
        times=time_steps,
        input_data=subduction_data,
        rotation_model=rotation_model,
        topology_features=topology_features,
        static_polygons=static_polygons,
        agegrid_dir=agegrid_dir,
        spreadrate_dir=spreadrate_dir,
        sedthick_dir=sedthick_dir,
        carbonate_dir=carbonate_dir,
        co2_dir=co2_dir,
        n_jobs=n_jobs,
        verbose=True,
    )

    subduction_data["plate_thickness (m)"] = plate_isotherm_depth(
        subduction_data["seafloor_age (Ma)"],
        maxiter=100,
    )

    subduction_data = calculate_water_thickness(data=subduction_data)
    subduction_data = calculate_carbon(subduction_data)
    subduction_data = calculate_slab_flux(subduction_data)
    subduction_data = calculate_slab_dip(subduction_data)

    subduction_data = extract_subducted_thickness(
        subduction_data,
        plate_reconstruction=plate_reconstruction,
        grid_resolution = grid_resolution,
    )

    subduction_data["sediment_flux (m^2/yr)"] = (
        subduction_data["sediment_thickness (m)"]
        * subduction_data["convergence_rate_orthogonal (cm/yr)"] * 1.0e-2
    ).clip(0.0, np.inf)

    subduction_data["carbon_flux (t/m/yr)"] = (
        subduction_data["total_carbon_density (t/m^2)"]
        * subduction_data["convergence_rate_orthogonal (cm/yr)"] * 1.0e-2
    ).clip(0.0, np.inf)

    subduction_data["water_flux (m^2/yr)"] = (
        subduction_data["total_water_thickness (m)"]
        * subduction_data["convergence_rate_orthogonal (cm/yr)"] * 1.0e-2
    ).clip(0.0, np.inf)

    subduction_data.to_csv(subduction_data_filename, index=False)

In [5]:
subduction_data_columns = subduction_data.columns.tolist()
features_plot = subduction_data_columns.copy()
features_plot.remove('lon')
features_plot.remove('lat')
features_plot.remove('age (Ma)')
features_plot.remove('subducting_plate_ID')
features_plot.remove('trench_plate_ID')

projection = ccrs.Mollweide(central_longitude=60)

### Visualisation

In [6]:
@interact
def show_map(time=time_steps, feature=features_plot):
    gplot.time = time
    
    agegrid_filename =  f'seafloor_age_{time}Ma.nc'
    agegrid_file = os.path.join(agegrid_dir, agegrid_filename)    
    agegrid = gplately.grids.read_netcdf_grid(agegrid_file)
    
    subduction_data_t = subduction_data[subduction_data["age (Ma)"] == time]

    fig = plt.figure(figsize=(16, 12))
    ax = fig.add_axes(
        [0.1, 0.1, 0.8, 0.8],
        projection=projection,
        facecolor=(plt.cm.colors.to_rgba('darkgray', alpha=0.5)),
    )
    ax.set_global()

    cax_feat = fig.add_axes([0.1, 0.12, 0.35, 0.02])
    cax_bg = fig.add_axes([0.55, 0.12, 0.35, 0.02])
    
    bg = gplot.plot_grid(ax, agegrid.data, cmap=ccm.lapaz_r, vmin=0, vmax=230, alpha=0.7, zorder=1)
    gplot.plot_coastlines(ax, facecolor='darkgray', edgecolor='none', zorder=2)
    gplot.plot_plate_motion_vectors(ax, spacingX=10, spacingY=10, normalise=True, alpha=0.1, zorder=3)

    feat = ax.scatter(subduction_data_t['lon'], subduction_data_t['lat'], 50, marker='.',
                      c=subduction_data_t[feature], cmap=ccm.hawaii_r, transform=ccrs.PlateCarree(), zorder=4)

    gplot.plot_ridges_and_transforms(ax, color='dimgray', linewidth=1.5, zorder=5)
    gplot.plot_trenches(ax, color='k', alpha=0.3, zorder=6)
    gplot.plot_subduction_teeth(ax, spacing=0.05, color='k', alpha=0.3, zorder=7)
    
    gl = ax.gridlines(crs=ccrs.PlateCarree(), draw_labels=True, x_inline=False, linewidth=1, color="gray", alpha=0.3, linestyle="--", zorder=8)
    
    ax.text(0.49,-0.03, '60°E', transform=ax.transAxes, fontsize=16)
    ax.text(0.46,-0.03, '0°', transform=ax.transAxes, fontsize=16)
    ax.text(0.40,-0.025, '60°W', transform=ax.transAxes, fontsize=16)
    
    gl.top_labels=False
    gl.bottom_labels=False
    
    gl.xlabel_style = {"size": 16}
    gl.ylabel_style = {"size": 16}
        
    cbar_feat = fig.colorbar(feat, cax=cax_feat, orientation="horizontal", extend='max')
    cbar_feat.set_label(format_feature_name(feature), fontsize=16, labelpad=10)
    cbar_feat.ax.tick_params(labelsize=16)

    cbar_bg = fig.colorbar(bg, cax=cax_bg, orientation="horizontal", extend='max')
    cbar_bg.set_label("Seafloor Age (Ma)", fontsize=16, labelpad=10)
    cbar_bg.set_ticks([0, 50, 100, 150, 200])
    cbar_bg.ax.tick_params(labelsize=16)
    
    # Define custom legend handles
    custom_handles = [
        Patch(facecolor='darkgray', edgecolor='none', label='Continental Crust'),
        Line2D([0], [0], color='dimgray', lw=2, label='Mid-Ocean Ridges/\nTransform Faults')
    ]

    # Add the custom legend to the plot
    ax.legend(handles=custom_handles, fontsize=16, loc='lower left', bbox_to_anchor=(0, -0.15))
    
    ax.set_title(f'{time} Ma', fontsize=25, y=1.04)
        
    plt.show()

interactive(children=(Dropdown(description='time', options=(0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, …